## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [40]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


In [41]:
MODEL = "gpt-4.1-nano"
HF_MODEL = "openai/gpt-oss-120b"  # OpenAI OSS equivalent by Meta
# HF_MODEL = "HuggingFaceH4/zephyr-7b-beta"  # OpenAI OSS equivalent by Meta
# HF_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # OpenAI OSS equivalent by Meta
DB_NAME = "vector_db"
load_dotenv(override=True)


True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [42]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [43]:
retriever = vectorstore.as_retriever()

In [ ]:
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
# The -q flag suppresses verbose output, 
# and using %pip (instead of !pip) ensures it installs into the same kernel environment
# the notebook is running in.
%pip install langchain-google-genai -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# curl "https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent" \
#   -H 'Content-Type: application/json' \
#   -H 'X-goog-api-key: KEY' \
#   -X POST \
#   -d '{
#     "contents": [
#       {
#         "parts": [
#           {
#             "text": "Explain how AI works in a few words"
#           }
#         ]
#       }
#     ]
#   }'

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_MODEL = "gemini-2.0-flash-lite"
google_api_key = os.getenv("GOOGLE_API_KEY")

# Gemini drop-in replacement for ChatOpenAI — same .invoke() interface
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=google_api_key,
    temperature=0,
)


## Can swap the HuggingFaceEndpoint + ChatHuggingFace for ChatOllama:

In [ ]:
# Ollama — if you have gpt-oss:20b pulled locally via Ollama,
# can swap the HuggingFaceEndpoint + ChatHuggingFace for ChatOllama:

# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="gpt-oss:20b", temperature=0)

# Ollama — local or remote
# If the remote server requires authentication, you can also pass custom headers
# via client_kwargs={"headers": {"Authorization": "Bearer <token>"}}
# Default base_url is http://localhost:11434
# To use a remote/public Ollama server, pass base_url explicitly:

# from langchain_ollama import ChatOllama
# llm = ChatOllama(
#     model="gpt-oss:20b",
#     base_url="https://your-ollama-host.com",  # remote Ollama URL
#     temperature=0,
# )

### HuggingFace Inference API — Serverless vs Dedicated Endpoints

| | URL format | When to use |
|---|---|---|
| **Serverless** | `api-inference.huggingface.co/models/{repo_id}` | Free tier, quick testing, quota-limited |
| **Dedicated** | `{id}.{region}.{cloud}.endpoints.huggingface.cloud` | Production, private, guaranteed throughput |

```python
# Serverless — just use repo_id (LangChain builds the URL internally)
HuggingFaceEndpoint(repo_id="HuggingFaceH4/zephyr-7b-beta", ...)

# Dedicated — use endpoint_url directly
HuggingFaceEndpoint(endpoint_url="https://xyz123.us-east-1.aws.endpoints.huggingface.cloud", ...)
```


In [27]:
from huggingface_hub import list_models, model_info

# List text-generation models with serverless inference enabled
models = list_models(
    filter="text-generation",
    inference="warm",       # "warm" = serverless API ready
    sort="likes",
    limit=20,
)
for m in models:
    print(m.id)
info = model_info("HuggingFaceH4/zephyr-7b-beta")
print(info.inference)   # "warm", "cold", or "stopped"

# warm → available now via serverless API
# cold → needs a few seconds to spin up
# stopped → not available serverlessly (need a dedicated endpoint or local pipeline)

deepseek-ai/DeepSeek-R1
meta-llama/Meta-Llama-3-8B
meta-llama/Llama-3.1-8B-Instruct
deepseek-ai/DeepSeek-V4-Pro
openai/gpt-oss-120b
openai/gpt-oss-20b
meta-llama/Meta-Llama-3-8B-Instruct
mistralai/Mistral-7B-v0.1
deepseek-ai/DeepSeek-V3
microsoft/phi-2
google/gemma-7b
mistralai/Mistral-7B-Instruct-v0.2
deepseek-ai/DeepSeek-V3-0324
Qwen/QwQ-32B
meta-llama/Llama-3.3-70B-Instruct
deepseek-ai/DeepSeek-R1-0528
meta-llama/Llama-3.2-1B
moonshotai/Kimi-K2-Instruct
meta-llama/Llama-3.1-8B
microsoft/phi-4
warm


In [25]:
# HuggingFace open-source drop-in replacement for ChatOpenAI
endpoint = HuggingFaceEndpoint(
    repo_id=HF_MODEL, # shared, free/quota-limited, use repo_id
    # endpoint_url="https://api-inference.huggingface.co/models/HuggingFaceH4/zephyr-7b-beta", # paid, for production
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,          # deterministic output (equivalent to temperature=0)
)
llm = ChatHuggingFace(llm=endpoint)

## Download a HuggingFace model locally and use it
Memory requirements: 7B models need ~14GB RAM/VRAM in fp16. Use torch_dtype=torch.float32 if you only have CPU (doubles memory, slower). For smaller footprint, try mistralai/Mistral-7B-Instruct-v0.3 or TinyLlama/TinyLlama-1.1B-Chat-v1.0.

In [ ]:
# ── Download a HuggingFace model locally and use it ──────────────────────────
#
# Step 1: Download model weights to disk (one-time, saved to ~/.cache/huggingface/hub/)
#         Run this in a terminal or uncomment below:
#
#   huggingface-cli download HuggingFaceH4/zephyr-7b-beta
#
# Or in Python:
#
# from huggingface_hub import snapshot_download
# local_path = snapshot_download(repo_id="HuggingFaceH4/zephyr-7b-beta")
# print(f"Model downloaded to: {local_path}")

# Step 2: Load the locally downloaded model via transformers pipeline
# (needs ~14GB RAM/VRAM for a 7B model in fp16 — skip if not available)

# from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
# from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
# import torch

# LOCAL_MODEL = "HuggingFaceH4/zephyr-7b-beta"  # uses cached download automatically

# tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
# model = AutoModelForCausalLM.from_pretrained(
#     LOCAL_MODEL,
#     torch_dtype=torch.float16,   # use fp16 to halve memory usage
#     device_map="auto",            # auto-selects GPU if available, else CPU
# )

# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,
#     do_sample=False,
# )

# # Wrap in LangChain — same .invoke() interface as ChatOpenAI / ChatGoogleGenerativeAI
# llm_local = HuggingFacePipeline(pipeline=pipe)
# llm = ChatHuggingFace(llm=llm_local)

# # Verify it works
# from langchain_core.messages import HumanMessage
# print(llm.invoke([HumanMessage(content="Who is Avery?")]).content)


### HuggingFaceEndpoint vs pipeline — what's the difference?

| | `HuggingFaceEndpoint` | `pipeline` (transformers) |
|---|---|---|
| **Where model runs** | HF Inference API (remote, HF servers) | Locally on your machine |
| **Model downloaded?** | No | Yes — weights saved to `~/.cache/huggingface/hub/` |
| **Requires HF token?** | Yes (for private/gated models or rate limits) | Only for gated models |
| **RAM/GPU needed?** | No | Yes — needs enough VRAM/RAM to load weights |
| **Latency** | Network round-trip | Fast once loaded (local inference) |
| **Cost** | HF Inference API credits/quota | Free (compute is yours) |

**Use `HuggingFaceEndpoint`** when you want a quick API call without local resources.  
**Use `pipeline`** when you want full local control, offline use, or low-latency inference.


In [ ]:
# pipeline approach — downloads model LOCALLY, runs inference on your machine
# (do NOT run this unless you have enough RAM/VRAM; 7B models need ~14GB in fp16)

# from transformers import pipeline
# from langchain_huggingface import HuggingFacePipeline

# pipe = pipeline(
#     "text-generation",
#     model=HF_MODEL,          # downloads weights to ~/.cache/huggingface/hub/
#     device_map="auto",        # auto-selects GPU if available, otherwise CPU
#     max_new_tokens=512,
#     do_sample=False,
# )
# llm_local = HuggingFacePipeline(pipeline=pipe)

# Same interface as HuggingFaceEndpoint — works with ChatHuggingFace too:
# from langchain_huggingface import ChatHuggingFace
# llm_local_chat = ChatHuggingFace(llm=llm_local)
# llm_local_chat.invoke([HumanMessage(content="Who is Avery?")])


### These LangChain objects implement the method `invoke()`

In [45]:
retriever.invoke("Who is Avery?")

[Document(id='c6190fd7-fe87-4502-a08c-82f62536c4fe', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [38]:
llm.invoke("Who is Avery?")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 9.806402049s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemin

## Time to put this together!

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!

## Can I provide tools to Ollama or HuggingFace models?

Yes, but with caveats depending on the backend.

---

### Ollama

Most modern Ollama models support tool calling natively (if the underlying model does — e.g. `llama3.1`, `mistral`, `qwen2.5`, `command-r`):

```python
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

@tool
def get_policy_info(policy_id: str) -> str:
    """Look up an insurance policy by ID."""
    return f"Policy {policy_id}: active, $500 deductible"

llm = ChatOllama(model="llama3.1", temperature=0)
llm_with_tools = llm.bind_tools([get_policy_info])

# LangChain handles the tool call/response loop
response = llm_with_tools.invoke("What's the deductible for policy POL-123?")
```

Check if your model supports it:
```bash
ollama show llama3.1 | grep tools
```

---

### HuggingFace (`HuggingFaceEndpoint` / serverless)

Only works if the model itself was trained for tool use (e.g. `Qwen/Qwen2.5-72B-Instruct`, `mistralai/Mixtral-8x7B-Instruct-v0.1`). `ChatHuggingFace` supports `.bind_tools()` with the same interface:

```python
llm_with_tools = llm.bind_tools([get_policy_info])  # llm = ChatHuggingFace(...)
```

However, not all models handle tool schemas correctly via the serverless API — smaller/older models will often ignore tool definitions or hallucinate calls.

---

### `HuggingFacePipeline` (local)

Tool use is **not supported** natively — you'd have to implement it manually using prompt engineering (inject the tool schema as JSON in the system prompt and parse the output).

---

### Summary

| Backend | Tool support | Reliability |
|---|---|---|
| `ChatOllama` (llama3.1+, qwen2.5+) | Yes, native | Good |
| `ChatHuggingFace` + serverless (Qwen2.5, Mixtral) | Yes, via `.bind_tools()` | Model-dependent |
| `HuggingFacePipeline` (local) | Manual only | Poor |
| `ChatOpenAI` / `ChatGoogleGenerativeAI` | Yes, native | Excellent |
